# Day 10 — CampusConnect Data Pipeline

Google Colab-first hands-on: validation, data transformation, testing, and outputs. The optional Azure section below demonstrates a two-step Azure ML pipeline.

Run cells in order. The core Colab exercise does not require Azure.

In [ ]:
%pip install pandas


In [ ]:
from pathlib import Path
import pandas as pd
import logging
from IPython.display import display


In [ ]:
# Create folders in the temporary Colab workspace.
Path("sample_data").mkdir(exist_ok=True)
Path("output").mkdir(exist_ok=True)

rows = [
    {"Student_ID": "S101", "Course": "Python", "Class_Date": "2026-10-12", "Classes_Attended": 9, "Classes_Held": 10},
    {"Student_ID": "S102", "Course": "Python", "Class_Date": "2026-10-12", "Classes_Attended": 8, "Classes_Held": 10},
    {"Student_ID": "S103", "Course": "Data Analytics", "Class_Date": "2026-10-12", "Classes_Attended": 10, "Classes_Held": 10},
    {"Student_ID": "S104", "Course": "Data Analytics", "Class_Date": "2026-10-12", "Classes_Attended": 6, "Classes_Held": 10},
    {"Student_ID": "", "Course": "Python", "Class_Date": "2026-10-12", "Classes_Attended": 7, "Classes_Held": 10},
    {"Student_ID": "S105", "Course": "Cloud Basics", "Class_Date": "2026-10-12", "Classes_Attended": 12, "Classes_Held": 10},
    {"Student_ID": "S106", "Course": "Cloud Basics", "Class_Date": "2026-10-12", "Classes_Attended": 5, "Classes_Held": 0},
    {"Student_ID": "S102", "Course": "Python", "Class_Date": "2026-10-12", "Classes_Attended": 8, "Classes_Held": 10},
    {"Student_ID": "S107", "Course": "Cloud Basics", "Class_Date": "2026-10-12", "Classes_Attended": 7, "Classes_Held": 10},
]

df_source = pd.DataFrame(rows)
input_path = Path("sample_data/attendance.csv")
df_source.to_csv(input_path, index=False)

print("Created:", input_path)
print("Number of rows:", len(df_source))
display(df_source)


In [ ]:
df = pd.read_csv("sample_data/attendance.csv")

print("Rows and columns:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())
print("\nData types and missing values:")
display(df.isna().sum().to_frame("Missing values"))
display(df.head())


In [ ]:
def run_attendance_pipeline(input_file="sample_data/attendance.csv",
                            output_dir="output"):
    input_file = Path(input_file)
    output_dir = Path(output_dir)

    if not input_file.exists():
        raise FileNotFoundError(f"Input file not found: {input_file}")

    data = pd.read_csv(input_file)

    required = [
        "Student_ID", "Course", "Class_Date",
        "Classes_Attended", "Classes_Held"
    ]
    missing_columns = set(required) - set(data.columns)
    if missing_columns:
        raise ValueError(
            f"Missing required columns: {sorted(missing_columns)}"
        )

    # Convert counts to numbers; invalid text becomes NaN.
    data["Classes_Attended"] = pd.to_numeric(
        data["Classes_Attended"], errors="coerce"
    )
    data["Classes_Held"] = pd.to_numeric(
        data["Classes_Held"], errors="coerce"
    )

    # Standardize required text fields.
    for col in ["Student_ID", "Course", "Class_Date"]:
        data[col] = data[col].astype("string").str.strip()
        data[col] = data[col].replace("", pd.NA)

    # Accumulate rejection reasons instead of silently dropping records.
    reasons = pd.Series("", index=data.index, dtype="string")

    def add_reason(mask, message):
        nonlocal reasons
        reasons.loc[mask] = reasons.loc[mask].where(
            reasons.loc[mask].eq(""),
            reasons.loc[mask] + "; "
        ) + message

    missing_values = data[required].isna().any(axis=1)
    add_reason(missing_values, "Missing required value")

    invalid_held = (
        data["Classes_Held"].notna()
        & (data["Classes_Held"] <= 0)
    )
    add_reason(invalid_held, "Classes_Held must be greater than 0")

    invalid_attended = (
        data["Classes_Attended"].notna()
        & data["Classes_Held"].notna()
        & (
            (data["Classes_Attended"] < 0)
            | (data["Classes_Attended"] > data["Classes_Held"])
        )
    )
    add_reason(
        invalid_attended,
        "Classes_Attended must be between 0 and Classes_Held"
    )

    duplicate_key = data.duplicated(
        subset=["Student_ID", "Course", "Class_Date"],
        keep="first"
    )
    add_reason(
        duplicate_key,
        "Duplicate Student_ID/Course/Class_Date"
    )

    rejected_mask = reasons.ne("")
    clean = data.loc[~rejected_mask].copy()
    rejected = data.loc[rejected_mask].copy()
    rejected["Rejection_Reason"] = reasons.loc[rejected_mask]

    clean["Attendance_Percentage"] = (
        clean["Classes_Attended"] / clean["Classes_Held"] * 100
    ).round(2)

    summary = (
        clean.groupby("Course", as_index=False)
        .agg(
            Valid_Records=("Student_ID", "count"),
            Average_Attendance_Percentage=("Attendance_Percentage", "mean"),
            Total_Classes_Attended=("Classes_Attended", "sum"),
            Total_Classes_Held=("Classes_Held", "sum"),
        )
    )
    summary["Average_Attendance_Percentage"] = (
        summary["Average_Attendance_Percentage"].round(2)
    )

    output_dir.mkdir(parents=True, exist_ok=True)
    clean.to_csv(output_dir / "clean_attendance.csv", index=False)
    rejected.to_csv(output_dir / "rejected_attendance.csv", index=False)
    summary.to_csv(output_dir / "course_summary.csv", index=False)

    logging.info("Input rows: %d", len(data))
    logging.info("Clean rows: %d", len(clean))
    logging.info("Rejected rows: %d", len(rejected))
    logging.info("Output directory: %s", output_dir.resolve())
    logging.info("Pipeline completed successfully.")

    return clean, rejected, summary

In [ ]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s: %(message)s"
)

clean, rejected, summary = run_attendance_pipeline()

print("\nCLEAN RECORDS")
display(clean)

print("\nREJECTED RECORDS")
display(rejected[["Student_ID", "Course", "Rejection_Reason"]])

print("\nCOURSE SUMMARY")
display(summary)

In [ ]:
assert len(df_source) == 9
assert len(clean) == 5
assert len(rejected) == 4
assert round(clean["Attendance_Percentage"].max(), 2) == 100.00

print("All basic checks passed.")
print("Input rows:", len(df_source))
print("Clean rows:", len(clean))
print("Rejected rows:", len(rejected))
print("Total rows accounted for:", len(clean) + len(rejected))


In [ ]:
clean_run_1, rejected_run_1, summary_run_1 = run_attendance_pipeline()
clean_run_2, rejected_run_2, summary_run_2 = run_attendance_pipeline()

print("Run 1 clean rows:", len(clean_run_1))
print("Run 2 clean rows:", len(clean_run_2))
print("Same clean output:", clean_run_1.equals(clean_run_2))

In [ ]:
try:
    run_attendance_pipeline("sample_data/does_not_exist.csv", "output/test")
except FileNotFoundError as error:
    print("Expected error caught:", error)


In [ ]:
bad_df = df_source.drop(columns=["Course"])
bad_path = Path("sample_data/attendance_missing_course.csv")
bad_df.to_csv(bad_path, index=False)

try:
    run_attendance_pipeline(str(bad_path), "output/bad_test")
except ValueError as error:
    print("Expected schema error caught:", error)


In [ ]:
from google.colab import files

files.download("output/clean_attendance.csv")
files.download("output/rejected_attendance.csv")
files.download("output/course_summary.csv")


In [ ]:
import shutil
from google.colab import files

zip_path = shutil.make_archive("campusconnect_outputs", "zip", "output")
files.download(zip_path)


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from pathlib import Path
import shutil

drive_output = Path("/content/drive/MyDrive/CampusConnect_Day10")
drive_output.mkdir(parents=True, exist_ok=True)

for file_path in Path("output").glob("*.csv"):
    shutil.copy2(file_path, drive_output / file_path.name)

print("Copied outputs to:", drive_output)


## Optional Azure ML extension

The Colab practical above is complete without Azure. Continue only with an authorized Azure ML workspace and approved compute. Use the workspace config file and never store passwords, tokens, or one-time sign-in codes in this notebook.

### 1. Connect to the correct Azure ML workspace

Download `config.json` from the intended Azure ML workspace's Overview page and upload it below. The workspace region is read from this file; do not hard-code the region. If Colab renames the upload to `config (1).json`, the code below handles it.

**Never commit your workspace config file, access tokens, or sign-in codes to GitHub.**

In [ ]:
from google.colab import files
import shutil
import json
from pathlib import Path

uploaded = files.upload()
json_files = [name for name in uploaded if name.lower().endswith(".json")]
if not json_files:
    raise FileNotFoundError("Upload the config.json downloaded from your Azure ML workspace.")

# Colab may rename a duplicate upload, e.g. config (1).json.
source_file = json_files[0]
shutil.copy(source_file, "config.json")

with open("config.json", "r", encoding="utf-8") as f:
    workspace_config = json.load(f)

print("Workspace config saved for this Colab session.")
print("Workspace name:", workspace_config.get("name", "(not listed in config)"))
print("Resource group:", workspace_config.get("resource_group", "(not listed in config)"))
print("Location:", workspace_config.get("location", "(not listed in config)"))

In [ ]:
%pip install -q azure-ai-ml azure-identity

In [ ]:
from azure.ai.ml import MLClient
from azure.identity import DeviceCodeCredential

credential = DeviceCodeCredential()
ml_client = MLClient.from_config(credential=credential, path="config.json")
workspace = ml_client.workspaces.get(ml_client.workspace_name)

print("Connected workspace:", workspace.name)
print("Resource group:", workspace.resource_group)
print("Workspace region:", workspace.location)
print("Use this workspace's region when checking Azure compute quota.")

### 2. Create the component scripts

The cleaning step creates clean and rejected records. The summary step consumes the clean file. This makes the dependency between steps explicit.

In [ ]:
from pathlib import Path
Path('src').mkdir(exist_ok=True)
print('Source folder ready:', Path('src').resolve())

In [ ]:
%%writefile src/clean_step.py
import argparse
from pathlib import Path
import pandas as pd

def main():
    p = argparse.ArgumentParser()
    p.add_argument('--input', required=True)
    p.add_argument('--clean_output', required=True)
    p.add_argument('--rejected_output', required=True)
    a = p.parse_args()
    d = pd.read_csv(a.input)
    required = ['Student_ID','Course','Class_Date','Classes_Attended','Classes_Held']
    missing = sorted(set(required) - set(d.columns))
    if missing:
        raise ValueError(f'Missing required columns: {missing}')
    for c in ['Student_ID','Course','Class_Date']:
        d[c] = d[c].astype('string').str.strip().replace('', pd.NA)
    d['Classes_Attended'] = pd.to_numeric(d['Classes_Attended'], errors='coerce')
    d['Classes_Held'] = pd.to_numeric(d['Classes_Held'], errors='coerce')
    reason = pd.Series('', index=d.index, dtype='string')
    def add(mask, msg):
        nonlocal reason
        reason.loc[mask] = reason.loc[mask].where(reason.loc[mask].eq(''), reason.loc[mask] + '; ') + msg
    add(d[required].isna().any(axis=1), 'Missing/invalid required value')
    add(d['Classes_Held'].notna() & (d['Classes_Held'] <= 0), 'Classes_Held must be greater than 0')
    bad = d['Classes_Attended'].notna() & d['Classes_Held'].notna() & ((d['Classes_Attended'] < 0) | (d['Classes_Attended'] > d['Classes_Held']))
    add(bad, 'Classes_Attended must be between 0 and Classes_Held')
    add(d.duplicated(subset=['Student_ID','Course','Class_Date'], keep='first'), 'Duplicate Student_ID/Course/Class_Date')
    mask = reason.ne('')
    clean = d.loc[~mask].copy()
    rejected = d.loc[mask].copy()
    rejected['Rejection_Reason'] = reason.loc[mask]
    clean['Attendance_Percentage'] = (clean['Classes_Attended'] / clean['Classes_Held'] * 100).round(2)
    Path(a.clean_output).parent.mkdir(parents=True, exist_ok=True)
    Path(a.rejected_output).parent.mkdir(parents=True, exist_ok=True)
    clean.to_csv(a.clean_output, index=False)
    rejected.to_csv(a.rejected_output, index=False)
    print(f'Input={len(d)}, clean={len(clean)}, rejected={len(rejected)}')

if __name__ == '__main__':
    main()

In [ ]:
%%writefile src/summarize_step.py
import argparse
from pathlib import Path
import pandas as pd

def main():
    p = argparse.ArgumentParser()
    p.add_argument('--clean_input', required=True)
    p.add_argument('--summary_output', required=True)
    a = p.parse_args()
    d = pd.read_csv(a.clean_input)
    s = d.groupby('Course', as_index=False).agg(
        Valid_Records=('Student_ID','count'),
        Average_Attendance_Percentage=('Attendance_Percentage','mean'),
        Total_Classes_Attended=('Classes_Attended','sum'),
        Total_Classes_Held=('Classes_Held','sum'))
    s['Average_Attendance_Percentage'] = s['Average_Attendance_Percentage'].round(2)
    Path(a.summary_output).parent.mkdir(parents=True, exist_ok=True)
    s.to_csv(a.summary_output, index=False)
    print('Summary rows:', len(s))

if __name__ == '__main__':
    main()

### 3. Register the input data and configure compute

This cell registers the sample CSV as a versioned Azure ML data asset. The component-definition cell below uses **Azure ML serverless compute** to avoid relying on the placeholder name `cpu-cluster`. Serverless execution must be supported and enabled for your workspace/subscription. If your organization requires a managed compute cluster instead, use the exact name of an existing cluster in this workspace.

Your workspace is in East US according to the latest workspace screenshot. The configuration file determines the workspace; changing the workspace region does not create compute quota automatically.

In [ ]:
from datetime import datetime
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

MY_ID = 'trainee01'  # change this to a unique lowercase ID
assert MY_ID.isalnum() and MY_ID.islower()
data_asset = Data(name=f'{MY_ID}-campusconnect-attendance', version=datetime.now().strftime('%Y%m%d%H%M%S'), type=AssetTypes.URI_FILE, path='sample_data/attendance.csv', description='Fictional CampusConnect attendance input')
registered = ml_client.data.create_or_update(data_asset)
print('Registered:', registered.name, registered.version)

In [ ]:
from azure.ai.ml import command, Input, Output, dsl
from azure.ai.ml.constants import AssetTypes

# Use Azure ML serverless compute if enabled for this workspace/subscription.
# If serverless is not enabled, replace "serverless" with the exact name
# of an existing Azure ML compute cluster in this same workspace.
COMPUTE_NAME = "serverless"
ENVIRONMENT = "azureml://registries/azureml/environments/sklearn-1.5/labels/latest"

clean_component = command(
    name="campusconnect_clean_attendance",
    display_name="CampusConnect - validate and clean",
    code="./src",
    command="python clean_step.py --input ${{inputs.attendance_csv}} --clean_output ${{outputs.clean_csv}} --rejected_output ${{outputs.rejected_csv}}",
    inputs={"attendance_csv": Input(type=AssetTypes.URI_FILE)},
    outputs={
        "clean_csv": Output(type=AssetTypes.URI_FILE),
        "rejected_csv": Output(type=AssetTypes.URI_FILE),
    },
    environment=ENVIRONMENT,
    compute=COMPUTE_NAME,
)

summary_component = command(
    name="campusconnect_course_summary",
    display_name="CampusConnect - summarize courses",
    code="./src",
    command="python summarize_step.py --clean_input ${{inputs.clean_csv}} --summary_output ${{outputs.summary_csv}}",
    inputs={"clean_csv": Input(type=AssetTypes.URI_FILE)},
    outputs={"summary_csv": Output(type=AssetTypes.URI_FILE)},
    environment=ENVIRONMENT,
    compute=COMPUTE_NAME,
)

print("Components defined. Compute target:", COMPUTE_NAME)
print("If submission reports serverless is unavailable, set COMPUTE_NAME to an existing cluster name and rerun this cell and all cells below it.")

### 4. Submit the connected two-step Azure ML pipeline

The summary step consumes the cleaning step's clean output. The submission cell below will now use the compute target configured above, instead of the nonexistent `cpu-cluster` placeholder.

If Azure rejects `serverless` because it is not enabled for this workspace or subscription, this is an Azure configuration/quota limitation, not a Python data-cleaning error. Use an existing compute cluster in this workspace or ask your administrator to enable serverless jobs/provide compute. Do not create a VM or cluster until quota and cost are confirmed.

In [ ]:
@dsl.pipeline(name='campusconnect-attendance-pipeline', description='Clean attendance and summarize courses')
def campusconnect_pipeline(attendance_csv):
    clean_step = clean_component(attendance_csv=attendance_csv)
    summary_step = summary_component(clean_csv=clean_step.outputs.clean_csv)
    return {'clean_csv': clean_step.outputs.clean_csv,
            'rejected_csv': clean_step.outputs.rejected_csv,
            'summary_csv': summary_step.outputs.summary_csv}

pipeline_job = campusconnect_pipeline(attendance_csv=Input(type=AssetTypes.URI_FILE, path=f'azureml:{registered.name}:{registered.version}'))
pipeline_job.settings.default_compute = COMPUTE_NAME
pipeline_job.display_name = f'{MY_ID}-campusconnect-two-step-pipeline'
pipeline_job.experiment_name = 'campusconnect-day10'
submitted = ml_client.jobs.create_or_update(pipeline_job)
print('Job:', submitted.name, '| initial status:', submitted.status)
print('Studio URL:', submitted.studio_url)

In [ ]:
import time
terminal = {'Completed', 'Failed', 'Canceled', 'NotResponding'}
while True:
    current = ml_client.jobs.get(submitted.name)
    status = str(current.status)
    print('Current status:', status)
    if status in terminal:
        break
    time.sleep(20)
if status != 'Completed':
    raise RuntimeError(f'Azure ML pipeline failed or stopped. Final status: {status}')
print('Pipeline completed. Inspect clean_csv, rejected_csv, and summary_csv in Azure ML Studio.')

## Scheduling and production readiness

- **Colab:** suitable for a guided, interactive lab; runtimes can disconnect and are not a dependable unattended scheduler.
- **Pipeline:** defines the processing steps and their dependencies.
- **Scheduler/orchestrator:** use an approved cloud scheduling service to trigger the Azure ML pipeline on a schedule. Configure identity, retries, monitoring, and notifications.
- Confirm compute costs and stop unused resources according to your organization's policy.

### Final checklist
- [ ] Uploaded the approved workspace config.json.
- [ ] Created both component scripts.
- [ ] Configured a real, authorized compute target.
- [ ] Registered the input data asset.
- [ ] Submitted the connected clean → summarize pipeline.
- [ ] Verified final status is Completed.
- [ ] Inspected clean, rejected, and summary outputs.
- [ ] Did not save credentials or one-time sign-in codes.